In [13]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report
)
import fasttext
import fasttext.util
import os
import urllib.request

In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [15]:
train_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/train.csv")
val_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/val.csv")
test_df = pd.read_csv("/kaggle/input/datasets/zahrapriyono/emotion-classification-in-e-commerce/Emotion-Classification-in-E-Commerce/data/processed/test.csv")
emotion_names = ['Happy', 'Love', 'Sadness', 'Fear', 'Anger']

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Label distribution (train):\n{train_df['label'].value_counts().sort_index()}")

Train: 24404 | Val: 3051 | Test: 3051
Label distribution (train):
label
0    14911
1     6847
2     1653
3      320
4      673
Name: count, dtype: int64


In [16]:
os.makedirs("../models/bilstm", exist_ok=True)

FT_MODEL_PATH = "../models/bilstm/cc.bn.300.bin"   # ← bn = Bengali

if not os.path.exists(FT_MODEL_PATH):
    print("Downloading Bengali FastText model...")
    fasttext.util.download_model('bn', if_exists='ignore')  # ← 'bn'
    import shutil
    shutil.move("cc.bn.300.bin", FT_MODEL_PATH)
    print("Done.")
else:
    print("FastText model already exists.")

ft_model = fasttext.load_model(FT_MODEL_PATH)
print(f"FastText (Bengali) loaded. Dim: {ft_model.get_dimension()}")

FastText model already exists.
FastText (Bengali) loaded. Dim: 300


In [17]:
# Build vocabulary from training data
from collections import Counter

import re
def tokenize(text):
    return re.findall(r'\b\w+\b', str(text).lower())

all_tokens = [tok for text in train_df['text'] for tok in tokenize(text)]
vocab_counter = Counter(all_tokens)
vocab = ['<PAD>', '<UNK>'] + [w for w, c in vocab_counter.most_common() if c >= 1]
word2idx = {w: i for i, w in enumerate(vocab)}

VOCAB_SIZE = len(vocab)
EMBED_DIM  = ft_model.get_dimension()   # 300
MAX_LEN    = 128

print(f"Vocabulary size: {VOCAB_SIZE}")

# Build embedding matrix using FastText vectors
embedding_matrix = np.zeros((VOCAB_SIZE, EMBED_DIM), dtype=np.float32)
for word, idx in word2idx.items():
    if word not in ('<PAD>', '<UNK>'):
        embedding_matrix[idx] = ft_model.get_word_vector(word)

print(f"Embedding matrix shape: {embedding_matrix.shape}")

Vocabulary size: 16284
Embedding matrix shape: (16284, 300)


In [18]:
def encode_text(text, word2idx, max_len):
    tokens = tokenize(text)[:max_len]
    ids = [word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>
    # Pad
    ids += [0] * (max_len - len(ids))
    return ids

class BiLSTMDataset(Dataset):
    def __init__(self, texts, labels):
        self.X = [encode_text(t, word2idx, MAX_LEN) for t in texts]
        self.y = labels

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.long),
            torch.tensor(self.y[idx],  dtype=torch.long)
        )

train_dataset = BiLSTMDataset(train_df['text'].tolist(), train_df['label'].tolist())
val_dataset   = BiLSTMDataset(val_df['text'].tolist(),   val_df['label'].tolist())
test_dataset  = BiLSTMDataset(test_df['text'].tolist(),  test_df['label'].tolist())

# Sample check
x, y = train_dataset[0]
print(f"Input shape: {x.shape}, Label: {y.item()}")

Input shape: torch.Size([128]), Label: 1


In [19]:
BATCH_SIZE = 64

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE)

print(f"Train batches: {len(train_loader)} | Val: {len(val_loader)} | Test: {len(test_loader)}")

Train batches: 382 | Val: 48 | Test: 48


In [20]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_size, num_classes,
                 embedding_matrix, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.embedding.weight = nn.Parameter(
            torch.tensor(embedding_matrix),
            requires_grad=True          # ← unfreeze
        )
        self.bilstm = nn.LSTM(
            embed_dim, hidden_size,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=dropout
        )
        self.attn_fc  = nn.Linear(hidden_size * 2, 1)   # ← attention
        self.dropout  = nn.Dropout(dropout)
        self.fc       = nn.Linear(hidden_size * 2, num_classes)

    def forward(self, x):
        emb = self.dropout(self.embedding(x))        # (B, L, E)
        out, _ = self.bilstm(emb)                    # (B, L, 2H)
        scores  = self.attn_fc(out)                  # (B, L, 1)
        weights = torch.softmax(scores, dim=1)       # (B, L, 1)
        context = (out * weights).sum(dim=1)         # (B, 2H)
        context = self.dropout(context)
        return self.fc(context)                      # (B, num_classes)

model = BiLSTMClassifier(
    vocab_size=VOCAB_SIZE,
    embed_dim=EMBED_DIM,
    hidden_size=128,
    num_classes=5,
    embedding_matrix=embedding_matrix,
    dropout=0.5
).to(device)

print(model)
print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

BiLSTMClassifier(
  (embedding): Embedding(16284, 300, padding_idx=0)
  (bilstm): LSTM(300, 128, num_layers=2, batch_first=True, dropout=0.5, bidirectional=True)
  (attn_fc): Linear(in_features=256, out_features=1, bias=True)
  (dropout): Dropout(p=0.5, inplace=False)
  (fc): Linear(in_features=256, out_features=5, bias=True)
)
Trainable params: 5,722,326


In [21]:
EPOCHS    = 10
LR        = 1e-3

# === IMPROVEMENT #1: Class weights ===
from sklearn.utils.class_weight import compute_class_weight

labels_array = train_df['label'].values
weights = compute_class_weight('balanced', classes=np.unique(labels_array), y=labels_array)
class_weights = torch.tensor(weights, dtype=torch.float).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)   # ← only this line changed

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

In [22]:
def evaluate(model, loader):
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            logits = model(X)
            preds  = logits.argmax(dim=-1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    return np.array(all_labels), np.array(all_preds)

best_val_f1 = 0
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0
    for X, y in train_loader:
        X, y = X.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits, y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item()

    # Validation
    val_labels, val_preds = evaluate(model, val_loader)
    _, _, val_f1, _ = precision_recall_fscore_support(
        val_labels, val_preds, average='weighted', zero_division=0
    )
    val_acc = accuracy_score(val_labels, val_preds)
    avg_loss = total_loss / len(train_loader)

    history.append({'epoch': epoch, 'loss': avg_loss, 'val_f1': val_f1, 'val_acc': val_acc})
    scheduler.step(1 - val_f1)

    print(f"Epoch {epoch:2d} | Loss: {avg_loss:.4f} | Val Acc: {val_acc:.4f} | Val F1: {val_f1:.4f}")

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), "../models/bilstm/bilstm_best.pt")
        print(f"  ✓ Best model saved (F1={val_f1:.4f})")

Epoch  1 | Loss: 1.3381 | Val Acc: 0.3353 | Val F1: 0.2307
  ✓ Best model saved (F1=0.2307)
Epoch  2 | Loss: 1.1492 | Val Acc: 0.5087 | Val F1: 0.5242
  ✓ Best model saved (F1=0.5242)
Epoch  3 | Loss: 1.0359 | Val Acc: 0.5880 | Val F1: 0.6013
  ✓ Best model saved (F1=0.6013)
Epoch  4 | Loss: 0.9236 | Val Acc: 0.5837 | Val F1: 0.6000
Epoch  5 | Loss: 0.8048 | Val Acc: 0.4808 | Val F1: 0.4752
Epoch  6 | Loss: 0.6612 | Val Acc: 0.5932 | Val F1: 0.6098
  ✓ Best model saved (F1=0.6098)
Epoch  7 | Loss: 0.6823 | Val Acc: 0.6637 | Val F1: 0.6477
  ✓ Best model saved (F1=0.6477)
Epoch  8 | Loss: 0.5312 | Val Acc: 0.6218 | Val F1: 0.6336
Epoch  9 | Loss: 0.4421 | Val Acc: 0.6257 | Val F1: 0.6350
Epoch 10 | Loss: 0.3974 | Val Acc: 0.6106 | Val F1: 0.6201


In [23]:
# Load best checkpoint
model.load_state_dict(torch.load("../models/bilstm/bilstm_best.pt", map_location=device))

test_labels, test_preds = evaluate(model, test_loader)

precision, recall, f1, _ = precision_recall_fscore_support(
    test_labels, test_preds, average='weighted', zero_division=0
)
acc = accuracy_score(test_labels, test_preds)

print("=== Test Set Results ===")
print(f"Accuracy:  {acc:.4f}")
print(f"F1:        {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print()
print("Classification Report:")
print(classification_report(test_labels, test_preds, target_names=emotion_names, zero_division=0))

=== Test Set Results ===
Accuracy:  0.6565
F1:        0.6393
Precision: 0.6438
Recall:    0.6565

Classification Report:
              precision    recall  f1-score   support

       Happy       0.71      0.84      0.77      1864
        Love       0.59      0.37      0.45       856
     Sadness       0.42      0.43      0.43       207
        Fear       0.09      0.10      0.10        40
       Anger       0.46      0.39      0.43        84

    accuracy                           0.66      3051
   macro avg       0.46      0.43      0.43      3051
weighted avg       0.64      0.66      0.64      3051



In [24]:
os.makedirs("../results", exist_ok=True)

df_results = pd.DataFrame({
    'text':          test_df['text'].values,
    'true_label_id': test_labels,
    'pred_label_id': test_preds,
    'true_emotion':  [emotion_names[i] for i in test_labels],
    'pred_emotion':  [emotion_names[i] for i in test_preds]
})
df_results.to_csv("../results/bilstm_results.csv", index=False)
print("Saved to ../results/bilstm_results.csv")

Saved to ../results/bilstm_results.csv


In [26]:
import os, torch

SAVE_PATH = "/kaggle/working/models/bilstm"
os.makedirs(SAVE_PATH, exist_ok=True)

torch.save({
    'model_state_dict': model.state_dict(),
    'vocab': vocab,
    'word2idx': word2idx,
    'embed_dim': EMBED_DIM,
    'hidden_size': 128,
    'num_classes': 5
}, f"{SAVE_PATH}/bilstm_model.pt")
print(f"BiLSTM saved to {SAVE_PATH}")

BiLSTM saved to /kaggle/working/models/bilstm


In [28]:
import pandas as pd, os

os.makedirs("/kaggle/working/results", exist_ok=True)

bilstm_results = {
    'model': 'BiLSTM-FastText',
    'accuracy': 0.66,
    'weighted_f1': 0.64,
    'macro_f1': 0.43,
    'f1_Happy': 0.77,
    'f1_Love': 0.45,
    'f1_Sadness': 0.43,
    'f1_Fear': 0.10,
    'f1_Anger': 0.43
}

pd.DataFrame([bilstm_results]).to_csv("/kaggle/working/results/bilstm_results.csv", index=False)
print("Saved: bilstm_results.csv")

Saved: bilstm_results.csv
